# Compare Scenarios — SRL-NER Sirah

Notebook ini membandingkan **5 skenario SRL-NER** post-run:
- **S1** — Baseline (fix THRESHOLD=0.9, no class weight, no contrastive, no augmentation)
- **S2a** — SCL (Strict Supervised Contrastive Learning) + Baseline
- **S2b** — JSCL (Jaccard Similarity Contrastive Learning, sentence-level) + Baseline
- **S3a** — Sentence Augmentation + S2a
- **S3b** — Sentence Augmentation + S2b

**Cara pakai:**
1. Pastikan tiap skenario sudah di-run dan model tersimpan di `done_running/<scenario>/output/models/<experiment_name>-iterative-6/` (atau `-base` kalau iter terakhir).
2. Run cell-cell di bawah berurutan. Cell `evaluate_all_scenarios()` akan: load model tiap skenario → run pada `test.csv` → cache metric ke `compare_cache.json`. Re-run notebook akan re-use cache.
3. Hasil: tabel ringkasan, plot pseudo-label dynamics, per-entity F1, confusion matrix subplot, P-R scatter.

**Reference:**
- Legacy compare notebook (E1+S1+S2 class weight/adaptive): `legacy_class_weight_adaptive/compare_scenarios.ipynb`
- Analisis tertulis: `analisis_skenario_S2_S3.md` (parent folder)

## 1. Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)

try:
    from seqeval.metrics import (
        f1_score as seq_f1, precision_score as seq_p, recall_score as seq_r,
        classification_report as seq_report,
    )
    _HAS_SEQEVAL = True
except ImportError:
    _HAS_SEQEVAL = False
    print('[warn] pip install seqeval untuk metric entity-level')

# Adjust DATASET_DIR untuk Colab (`/content/drive/MyDrive/TA-Sirah/`) atau lokal
ROOT = Path('.').resolve()
DONE_DIR = ROOT   # asumsikan notebook ini di done_running/
DATASET_DIR = ROOT.parents[3] / 'data' / 'result' / 'pseudo-labelling' / 'SRL-NER'
if not DATASET_DIR.exists():
    print(f'[warn] dataset dir not found: {DATASET_DIR}')
    print('       set DATASET_DIR ke lokasi train.csv / test.csv kamu')

SCENARIOS = {
    'S1 (baseline)':   {'dir': 'S1_baseline',    'experiment': 'bert-only-sirah-ner'},
    'S2a (SCL)':       {'dir': 'S2a_scl',        'experiment': 'bert-only-sirah-ner-S2a-scl'},
    'S2b (JSCL)':      {'dir': 'S2b_jscl',       'experiment': 'bert-only-sirah-ner-S2b-jscl'},
    'S3a (SCL+Aug)':   {'dir': 'S3a_scl_aug',    'experiment': 'bert-only-sirah-ner-S3a-scl-aug'},
    'S3b (JSCL+Aug)':  {'dir': 'S3b_jscl_aug',   'experiment': 'bert-only-sirah-ner-S3b-jscl-aug'},
}

CACHE_PATH = DONE_DIR / 'compare_cache.json'
MINOR_LABELS = {'B-EVENT', 'I-EVENT', 'I-LOCATION'}

print('Scenarios:')
for name, cfg in SCENARIOS.items():
    p = DONE_DIR / cfg['dir']
    print(f'  {name:<20} -> {p}  {"OK" if p.exists() else "MISSING"}')

## 2. Load test data

In [ ]:
df_test = pd.read_csv(DATASET_DIR / 'test.csv')
df_test['token'] = df_test['token'].astype(str)
# Notebook convention: replace '-' with '_' in label (B-PERSON -> B_PERSON)
df_test['label_orig'] = df_test['label'].copy()
df_test['label'] = df_test['label'].apply(lambda x: x.replace('-', '_'))

print(f'Test set: {len(df_test)} token, {df_test["text_id"].nunique()} kalimat')
print('\nDistribusi label test:')
print(df_test['label_orig'].value_counts())

## 3. Helper: evaluate one scenario

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm


def find_latest_iteration(scenario_dir: Path, experiment: str) -> Path | None:
    """Cari folder model iter tertinggi (config.json exists), fallback ke -base."""
    models_dir = scenario_dir / 'output' / 'models'
    if not models_dir.exists():
        return None
    # Coba iter 6 → 5 → ... → 2 → base
    for i in range(6, 1, -1):
        cand = models_dir / f'{experiment}-iterative-{i}'
        if (cand / 'config.json').exists():
            return cand
    base = models_dir / f'{experiment}-base'
    if (base / 'config.json').exists():
        return base
    return None


def extract_entities_from_result(tokens, result):
    """Map pipeline output → BIO labels per token (BIO scheme)."""
    out = []
    cur = 0
    prev_span = None
    for tok in tokens:
        hit = None
        for idx, ent in enumerate(result):
            if ent['start'] <= cur < ent['end']:
                hit = idx
                break
        if hit is None:
            out.append('O')
            prev_span = None
        else:
            group = result[hit]['entity_group']
            if group.startswith(('B_', 'I_')):
                out.append(group)
            else:
                out.append(f'B_{group}' if hit != prev_span else f'I_{group}')
            prev_span = hit
        cur += len(tok) + 1
    return out


def evaluate_scenario(name: str, cfg: dict, df_test: pd.DataFrame) -> dict | None:
    """
    Load model dari `done_running/<dir>/output/models/<experiment>-iterative-N/`,
    run pipeline pada df_test, compute metrics.
    """
    scenario_dir = DONE_DIR / cfg['dir']
    model_path = find_latest_iteration(scenario_dir, cfg['experiment'])
    if model_path is None:
        print(f'  [SKIP {name}] no model found in {scenario_dir}/output/models/')
        return None
    print(f'  [run  {name}] model: {model_path.name}')

    ner = pipeline('token-classification', model=str(model_path), aggregation_strategy='simple')
    predicted = []
    for tid in tqdm(df_test['text_id'].unique(), leave=False):
        toks = df_test[df_test['text_id'] == tid]['token'].tolist()
        text = ' '.join(toks)
        result = ner(text)
        predicted.extend(extract_entities_from_result(toks, result))

    y_true = df_test['label'].tolist()
    y_pred = predicted

    # Token-level metrics
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    labels_no_O = sorted({l for l in y_true if l != 'O'})
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels_no_O, average='macro', zero_division=0)

    # Per-label F1
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    out = {
        'name': name, 'model_path': str(model_path),
        'f1_weighted_token': float(f_w),
        'precision_weighted_token': float(p_w),
        'recall_weighted_token': float(r_w),
        'f1_macro_no_O': float(f_m),
        'per_label': {k: v for k, v in rep.items() if isinstance(v, dict) and k not in ('accuracy', 'macro avg', 'weighted avg')},
    }

    # Entity-level seqeval (kalau available)
    if _HAS_SEQEVAL:
        # Group y_true/y_pred per kalimat
        df_pred = df_test.copy()
        df_pred['predicted'] = y_pred
        sents_true = df_pred.groupby('text_id')['label'].apply(list).tolist()
        sents_pred = df_pred.groupby('text_id')['predicted'].apply(list).tolist()
        # seqeval butuh format 'B-X' (dash), kita pakai 'B_X' (underscore) — convert
        sents_true_dash = [[lab.replace('_', '-') for lab in s] for s in sents_true]
        sents_pred_dash = [[lab.replace('_', '-') for lab in s] for s in sents_pred]
        try:
            out['f1_entity_seqeval'] = float(seq_f1(sents_true_dash, sents_pred_dash))
            out['precision_entity_seqeval'] = float(seq_p(sents_true_dash, sents_pred_dash))
            out['recall_entity_seqeval'] = float(seq_r(sents_true_dash, sents_pred_dash))
        except Exception as e:
            print(f'    seqeval error: {e}')

    # Save predictions untuk plot confusion matrix nanti
    out['y_true'] = y_true
    out['y_pred'] = y_pred
    return out

## 4. Evaluate semua skenario (atau load dari cache)

In [ ]:
# Set FORCE_RECOMPUTE = True kalau mau re-run inference (mis. setelah re-train)
FORCE_RECOMPUTE = False

if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    print(f'[cache] loading {CACHE_PATH}')
    with open(CACHE_PATH, encoding='utf-8') as f:
        results = json.load(f)
    results = {k: v for k, v in results.items() if v is not None}
else:
    results = {}
    for name, cfg in SCENARIOS.items():
        r = evaluate_scenario(name, cfg, df_test)
        results[name] = r
    # cache (drop y_true/y_pred sebelum save supaya JSON tidak gede)
    to_save = {}
    for k, v in results.items():
        if v is None:
            to_save[k] = None
            continue
        cached = {kk: vv for kk, vv in v.items() if kk not in ('y_true', 'y_pred')}
        to_save[k] = cached
    with open(CACHE_PATH, 'w', encoding='utf-8') as f:
        json.dump(to_save, f, indent=2, ensure_ascii=False)
    print(f'[cache] saved {CACHE_PATH}')

print(f'\nAvailable results: {list(results.keys())}')

## 5. Tabel ringkasan metric

In [ ]:
rows = []
for name, r in results.items():
    if r is None:
        rows.append({'scenario': name, 'status': 'NOT RUN'})
        continue
    rows.append({
        'scenario': name,
        'F1_token_weighted': r.get('f1_weighted_token'),
        'F1_macro_no_O':     r.get('f1_macro_no_O'),
        'F1_entity_seqeval': r.get('f1_entity_seqeval'),
        'Precision_entity':  r.get('precision_entity_seqeval'),
        'Recall_entity':     r.get('recall_entity_seqeval'),
    })
summary = pd.DataFrame(rows)
summary.to_csv(DONE_DIR / 'summary_comparison_5skenarios.csv', index=False)
summary

## 6. Per-entity F1 (PERSON / LOCATION / TIME / EVENT)

In [ ]:
rows_pe = []
for name, r in results.items():
    if r is None:
        continue
    per_label = r.get('per_label', {})
    # Group B_X + I_X → entity-level F1 average
    for etype in ('PERSON', 'LOCATION', 'TIME', 'EVENT'):
        b = per_label.get(f'B_{etype}', {})
        i = per_label.get(f'I_{etype}', {})
        support = b.get('support', 0) + i.get('support', 0)
        if support == 0:
            continue
        f1_b = b.get('f1-score', np.nan)
        f1_i = i.get('f1-score', np.nan)
        # Support-weighted average over B + I
        if b.get('support', 0) and i.get('support', 0):
            f1_avg = (f1_b * b['support'] + f1_i * i['support']) / support
        else:
            f1_avg = f1_b if b.get('support', 0) else f1_i
        rows_pe.append({
            'scenario': name,
            'entity':   etype,
            'f1':       f1_avg,
            'support':  support,
        })
per_entity = pd.DataFrame(rows_pe)
per_entity.to_csv(DONE_DIR / 'per_entity_f1_5skenarios.csv', index=False)
per_entity_pivot = per_entity.pivot(index='entity', columns='scenario', values='f1')
print(per_entity_pivot.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
per_entity_pivot.plot(kind='bar', ax=ax)
ax.set_ylabel('F1 score (entity-level)')
ax.set_title('Per-entity F1 — S1 vs S2a/b vs S3a/b')
ax.legend(loc='lower left', bbox_to_anchor=(1, 0))
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(DONE_DIR / 'fig_per_entity_5skenarios.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. F1 EVENT spotlight (kelas paling minoritas)

In [ ]:
event_df = per_entity[per_entity['entity'] == 'EVENT'].copy()
print(event_df[['scenario', 'f1', 'support']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(event_df['scenario'], event_df['f1'], color=['#888'] + ['#2980b9']*2 + ['#27ae60']*2)
ax.set_ylabel('F1 EVENT')
ax.set_title('F1 EVENT — kelas paling minoritas (lihat dampak contrastive + augmentation)')
ax.set_ylim(0, 1)
for i, (sc, v) in enumerate(zip(event_df['scenario'], event_df['f1'])):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(DONE_DIR / 'fig_event_spotlight_5skenarios.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Pseudo-label dynamics (n_above per iter)

Pre-requisite: tiap scenario folder punya `output/evaluation/iteration_log.csv` (atau
`iteration_log_<experiment_name>.csv`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, cfg in SCENARIOS.items():
    eval_dir = DONE_DIR / cfg['dir'] / 'output' / 'evaluation'
    if not eval_dir.exists():
        continue
    # Coba 2 nama file (legacy vs S2/S3 yang punya suffix experiment_name)
    candidates = list(eval_dir.glob('iteration_log*.csv'))
    if not candidates:
        print(f'  [skip {name}] no iteration_log csv found')
        continue
    df_iter = pd.read_csv(candidates[0])
    ax.plot(df_iter['iter'], df_iter['n_above'], marker='o', label=name)
ax.set_xlabel('Iterasi')
ax.set_ylabel('n_above (pseudo-label baru per iter)')
ax.set_title('Dinamika Self-Training — pseudo-label per iter')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DONE_DIR / 'fig_pseudo_per_iter_5skenarios.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Precision-Recall trade-off (entity-level)

Scatter precision vs recall — kalau iso-F1 contour, posisi atas-kanan = pareto frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for name, r in results.items():
    if r is None or 'precision_entity_seqeval' not in r:
        continue
    p, rec, f1 = r['precision_entity_seqeval'], r['recall_entity_seqeval'], r.get('f1_entity_seqeval', 0)
    ax.scatter(p, rec, s=120, label=f'{name} (F1={f1:.3f})')
    ax.annotate(name.split(' ')[0], (p, rec), textcoords='offset points', xytext=(8, 8))

# iso-F1 contours
f1_levels = [0.7, 0.8, 0.85, 0.9, 0.95]
x = np.linspace(0.5, 1.0, 100)
for f1 in f1_levels:
    y = f1 * x / (2 * x - f1)
    mask = (y > 0) & (y < 1)
    ax.plot(x[mask], y[mask], '--', alpha=0.3, color='gray')
    if any(mask):
        ax.text(x[mask][-1], y[mask][-1], f'F1={f1}', alpha=0.5, fontsize=8)

ax.set_xlabel('Precision (entity-level seqeval)')
ax.set_ylabel('Recall (entity-level seqeval)')
ax.set_title('P-R Trade-off')
ax.set_xlim(0.5, 1.02)
ax.set_ylim(0.5, 1.02)
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DONE_DIR / 'fig_pr_tradeoff_5skenarios.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Confusion matrix per skenario (subplot)

In [ ]:
active = [(name, r) for name, r in results.items() if r is not None and 'y_true' in r]
if not active:
    print('Confusion matrix butuh y_true/y_pred — rerun cell #4 dengan FORCE_RECOMPUTE=True')
else:
    n = len(active)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_2d(axes).flatten()
    label_list = sorted({l for _, r in active for l in r['y_true']})
    for ax, (name, r) in zip(axes, active):
        cm = confusion_matrix(r['y_true'], r['y_pred'], labels=label_list)
        sns.heatmap(cm, annot=True, fmt='g', ax=ax, cmap='Oranges',
                    xticklabels=label_list, yticklabels=label_list, cbar=False)
        ax.set_title(name)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='x', labelrotation=90)
        ax.tick_params(axis='y', labelrotation=0)
    for ax in axes[len(active):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(DONE_DIR / 'fig_confusion_5skenarios.png', dpi=120, bbox_inches='tight')
    plt.show()

## 11. Verdict

Setelah semua plot ke-generate, isi verdict di sini berdasarkan:
1. **F1 entity-level (seqeval)** — metrik standar literatur NER. Skenario pemenang.
2. **F1 EVENT** — kelas paling minoritas, target utama revisi Bu Diana.
3. **Precision-Recall trade-off** — kualitas pseudo-label (precision tinggi = clean output untuk KG downstream).
4. **Marginal contribution per layer:**
   - ΔF1(S2a − S1) = efek SCL
   - ΔF1(S2b − S1) = efek JSCL
   - ΔF1(S3a − S2a) = efek augmentation di atas SCL
   - ΔF1(S3b − S2b) = efek augmentation di atas JSCL

**Template verdict (isi setelah lihat plot):**
- Best F1 entity-level: __
- Best F1 EVENT: __
- Best balanced precision-recall: __
- Pilihan inference final: __ (alasan: __)

Tulis ulang verdict di `../analisis_skenario_S2_S3.md`.